# Google Colab SSH Setup for Claude Code

This notebook sets up an SSH server on Colab and exposes it via **cloudflared** tunnel.
No account or credit card needed.

## Prerequisites

1. **GPU runtime**: Runtime → Change runtime type → T4 GPU
2. **Local machine**: needs `cloudflared` installed (the notebook prints install instructions)

## Usage

1. Run all cells
2. Copy the SSH connection info and give it to Claude Code

## Warnings

- SSH on Colab is a grey area in Google's ToS — use at your own risk
- Colab Pro sessions last up to ~24h; idle timeout ~90min
- All state is lost when the runtime restarts

In [ ]:
#@title 1. Install & Configure SSH + cloudflared { display-mode: "form" }

import secrets, subprocess, re, pathlib, time

# --- Install openssh-server ---
print("[1/4] Installing openssh-server...")
subprocess.run(
    ["apt-get", "install", "-qq", "-o=Dpkg::Use-Pty=0", "openssh-server"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True,
)

# --- Generate root password ---
print("[2/4] Setting root password...")
password = secrets.token_urlsafe(16)
subprocess.run(
    ["chpasswd"],
    input=f"root:{password}".encode(),
    check=True,
)

# --- Configure sshd ---
print("[3/4] Configuring SSH daemon...")
sshd_config = pathlib.Path("/etc/ssh/sshd_config")
config_text = sshd_config.read_text()
for key in ["PermitRootLogin", "PasswordAuthentication"]:
    config_text = re.sub(rf"^#?\s*{key}\s+.*$", "", config_text, flags=re.MULTILINE)
config_text += "\nPermitRootLogin yes\nPasswordAuthentication yes\n"
sshd_config.write_text(config_text)

pathlib.Path("/var/run/sshd").mkdir(parents=True, exist_ok=True)
subprocess.run(["/usr/sbin/sshd"], check=True)
print("  SSH daemon started.")

# --- Install cloudflared & start tunnel ---
print("[4/4] Installing cloudflared and starting tunnel...")
subprocess.run(
    ["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"],
    check=True,
)
subprocess.run(
    ["dpkg", "-i", "cloudflared-linux-amd64.deb"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True,
)

# Start cloudflared in background, capture the tunnel URL from stderr
import threading, queue

url_queue = queue.Queue()

def run_cloudflared():
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "ssh://localhost:22", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    # Read stderr line by line to find the tunnel URL
    for line in iter(proc.stderr.readline, b""):
        text = line.decode("utf-8", errors="replace")
        # cloudflared prints the URL like: https://xxx.trycloudflare.com
        match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", text)
        if match:
            url_queue.put(match.group(1))

t = threading.Thread(target=run_cloudflared, daemon=True)
t.start()

# Wait for the URL (timeout 30s)
try:
    tunnel_url = url_queue.get(timeout=30)
except queue.Empty:
    raise RuntimeError("Timed out waiting for cloudflared tunnel URL")

hostname = tunnel_url.replace("https://", "")

print()
print("=" * 60)
print("  SSH CONNECTION INFO (via cloudflared)")
print("=" * 60)
print(f"  Hostname: {hostname}")
print(f"  User:     root")
print(f"  Password: {password}")
print()
print("  Connect command (run on local machine):")
print(f'  ssh -o StrictHostKeyChecking=no -o ProxyCommand="cloudflared access ssh --hostname %h" root@{hostname}')
print()
print("  If cloudflared is not installed locally:")
print("    # Linux x86_64")
print("    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
print("=" * 60)
print()
print("Copy the info above and paste it to Claude Code.")

In [ ]:
#@title 2. Verify GPU { display-mode: "form" }
!nvidia-smi

In [ ]:
#@title 3. Keep-alive (run this to prevent idle timeout) { display-mode: "form" }
#@markdown This cell runs an infinite loop that prints a heartbeat every 5 minutes
#@markdown to prevent the Colab runtime from disconnecting due to inactivity.
#@markdown
#@markdown **Stop it manually** when you're done (click the stop button).

import time
from datetime import datetime

print("Keep-alive started. Press the stop button to end.")
while True:
    print(f"  heartbeat: {datetime.now().strftime('%H:%M:%S')}")
    time.sleep(300)